# An inflation model for assessing LPI capital requirements

## 1. Introduction

Insurers and in particular Bulk Annuity Purchase providers need a model to calculate inflation risk capital for LPI liabilities. 
The model needs to be flexible enough to accommodate ALM purposes; that is to identify the balance sheet impact from various inflation hedging strategies.

This note has been written with support from a large language model (or AI). The findings from working with AI are summarised in the last section but do not interfere or are mentioned in previous sections. This is because we feel it is appropriate that the author(s) bear the responsibility of the work and the choices made at various steps of the construction process. As such, the note has two distinct parts; the appropriately constructed model and the conclusions from working with AI.

This note is structured as follows: 
- section 2 discusses key requirements/criteria to be satisfied for a model that is fit for LPI capital requriments 
- section 3 has a brief look at CPI
- section 4 looks at candidate models and discusses targeting Intra-path volatility
- section 5 describes the specification of the chosen model
- section 6 shows simulated paths from the chosen model
- section 7 explains how AI was used to support this work and some of the author's thoughts on using AI
- and finally, section 8 summarises and concludes the note.

## 2. Criteria for an LPI inflation model and choosing a dataset

Τhe key criteria considered here in choosing a model are: 
- is parsimonous/ as simple as possible, based on the BIC (Bayes Information Criterion) statistic.
- it makes best use of all the available data, filtering out data that may no longer be relevant
- aligns with current expectations of how the risk variable will behave going forward.
- it is fit for purpose, which here is to estimate the capital for an LPI portfolio. 
- requires minimal expert intervention/judgement
- is modular and can be easily combined and be consistent with models for other risks of the company.

We consider a few of the above points further.

### 2.1 Model is fit for purpose. 

#### 2.1.1 Intra-path volatility (IPV)
Since there are various formats of LPI ([0,5] being one of the most popular ones but there are many) we need a model that allows inflation to behave as "erratically" as it has done so far. In that sense, a smooth model (for example a Bank of England inflation expectation curve) might not be entirely appropriate. Such a curve may result in a floor-ceiling being "hit" over consecutive years for long periods of time. That would be fine if historic data illustrate heteroscedasticity or there are strong grounds to expect such to arise in the future. If inflation is not heteroscedastic, however, the model will point to a strategy that may not be backed by historic inflation behaviour, resuting in over- or under-hedging the risk.

Failing to target the IPV can be significant for LPI, as the intra-simulation volatility is the key quantity that will determine capital in later steps. Overstating or understating IPV can also be costly. It is of course legitimate to apply a judgement and target an IPV that is not consistent with history, based on current views for example. 

#### 2.1.2 Annual metric
LPI is an annual metric. So we need to ensure that our model has features that resemble correctly annualised (and not monthly, for example) historic inflation. To that end one should consider annual data in both overlapping and non-overlapping formats. Nevertheless relevant historic data are available only since 1988 - before that a significantly different methodology was used as well as the items that participated in the index. From 1988 we only get 37 annual data points which is not too much if our projection horizon is 40 years. Nevertheless it can give us a good indication of the intra-path inflation volatility, which is an important quantity in modelling LPI. 

From the above points it becomes apparent that annual inflation rates and annualised volatility of year-on-year inflation are the key quantities one should attempt to target.

### 2.2 Data choices

This brings us to another point, of making best use of all available data. In our research we identified 4 data sources; 3 freely available and one commercial. They are as follows:
1. Historical statistics of inflation at month ends, such as CPI, RPI, CPIH, etc (published by ONS)
2. History of inflation expectations as manifested by the Bank of England inflation curves. Said curves reflect the difference between nominal and inflation-linked gilts after a smoothing process. (freely available)
3. Inflation expectations from derivatives market, found for example in the DTCC Trade directory (free)
4. Information from inflation derivatives tracked on a daily basis (Bloomberg - commercial). That information can be used to extract both level and expected volatility of inflation at future periods.


### 2.3 Model choices
There is abudance of literature of models on inflation (see 2,3,5,6,9 and 10), but they are mainly on the pricing side of the risk. Many of these models attempt to capture the volatility surface implied by market prices. There are however much fewer examples/models for risk and capital management (11,12,14). While desirable that capital and pricing models are consistent, given the different purposes of the two, one could legitimately justify use of different data for capital (history) and for pricing (current market expectations on future inflation). It is also the case that attempting to force a volatility structure seen in the market in a capital model may be unnecessary and introduces unwarranted complexity that is not supported from historic inflation behaviour.

For capital purposes we want to calculate the present value of assets and liabilities that are many years away. In this note we consider a horizon of 40 years. We need to be able to project annual inflation paths over that horizon.

It is important to consider which inflation features we should project forward. We are interested in: 
-    autocorrelation 
-    distribution of residuals - do they look normal, or fat-tailed? 
-    heteroscedasticity - does inflation remain extreme for long periods?
-    Intra-path volatility (IPV) - given that we will be simulating inflation paths, do the demonstrate similar volatility levels to ones seen before? 


### 2.4 Key Expert judgements
We note at this point that it is perfectly legitimate to apply judgements to capture current information that is potentially not captured in this model. For example, views on potential impacts from climate change, or Bank of England losing its independence in setting monetary policy. Such information may come from news bulletins, data on paid platforms (e.g. Bloomberg) or the Bank of England published inflation curves. These are potentially valid influences to this model and are briefly discussed in a later section, however we opted for a simple model that does not capture explicitly any of the above. Tweaking the proposed model to allow for further influences such as the ones mentioned here could be the topic of a later note.

At this point we need to consider a couple of further criteria, the need for simplicity and (in parallel) the desire to minimise expert judgement in the model. While historical inflation figures cannot be disputed, future expectations may reflect supply and demand dynamics as well rational expectations of shifts not happened historically. And it is not straightforward to distinguish between the two. See also [paper by BoE]. We therefore opted for a model based on historic data and for illustration purposes we based it on CPI. 

Finally, as discussed above, there is limited annualised inflation data, but significantly more monthly data. While using monthly data presents the additional issue of annualisation, it was felt that the higher volume of information would result in a more robust model than a model based on a more limited data series. Using historic data means we need to use a time-series methodology and we need to replicate as best as possible the behaviour of inflation over time, considering matters of autocorrelation and heteroscedasticity as previously mentioned.

It is worth noting that a solution via distributions is also feasible (e.g. by applying PCA to BoE inflation curves) and steps aside the time-series issues, however it suffers from being fit for purpose for LPI modelling (see for example 16). 

### 2.5 Chosen Dataset
As such, the model built here is based on historical data and in particular the **UK monthly CPI index since 1988**. The all-inclusive index is chosen, but the analysis could very easily be based on the RPI index or some other inflation variant (e.g. CPIH). It is also plausible that a different model would be chosen for a different index, but this work is indicative. We also do not discuss whether CPI, RPI, or indeed some mixture of the two might be suitable. 

### 2.6 Limitations
Arguably most LPI contracts are still anchored on RPI and probably RPI and CPI are not 100% correlated, but while this analysis is based on CPI, it could very easily be based on RPI. This might be the topic of a future note.

We should also note that the chosen model is not built on top of or in parallel with an interest rates model. The relationship between inflation and interest rates is very important and should be examined and possibly integrated with this work but is not considered here. Again, this could be the topic of a further note, as indeed there is strong correlation of Central Bank actions and inflation behaviour. 

## 3. Looking at CPI
Before we dive into models, lets have a look at what inflation looked historically.

<img src="CPI1988toYE25.png" width="800">

The chart shows that monthly inflation is mean reverting and stationary. While not clearly evident in the above chart, CPI demonstrates strongly seasonal behaviour. Annualised inflation with overlaps is (as expected) more volatile than annual inflation with no overlap (orange line). However basing a model on overlapping data series will overstate true autocorrelation of the underlying.

## 4. Candidate models

So we start with investigating the following models: 

- ARIMA, ARMA
- ARMA + GARCH

For each of the above models we considered 3 distributions for the residuals; Normal, Negative inverse gaussian (NIG) and t. The t distribution has infinite variance when there are less than 2 degrees of freedom.

We applied these models on monthly data ($\Delta$ logCPI) after allowing for seasonality through explicit seasonal components (seasonal differencing is equally valid and more parsimonous). We also considered the above models on annual non-overlapping data for different starting months (with no seasonality adjustment). 

### 4.1 Targeting the Intra-path volatility
As LPI capital will be influenced by the frequency by which inflation caps and floors are "hit" within a single simulation path, it is important to ensure that the volatility of inflation within each such path is correctly accounted for. We therefore define the intra-path volatility or IPV as the (sample) standard deviation of annualised inflation rates within a historic path. 

The intra-path volatility that we wish to target is a quantity that is based on limited data. Our (relevant) dataset only expands over 37 years - this provides with 12 estimates, one for each month of the year. It is also possible to consider this quantity on shorter paths - we decided to consider paths between 30 and 37 years for all possible periods and starting months. This resulted in the following chart.
<img src="IPV_historicbootstrapped.png" width="800">

<!-- ![Figure 2: Historic IPV bootstrapped statistics](IPV_historicbootstrapped.png) -->

As expected the dispersion of the statistics reduces in line with the reduction of the number of data-points for increasing periods. There is a clear trend that indicates that the IPV increases with increasing number of years on average and becomes more stable for 36 and 37 years, where we have only 24 and 12 paths to consider respectively. We deemed it prudent to disregard the flattening trend of the mean over the last two years and extrapolate the path between paths of 30 to 35 years long. This gives that:

$ \mu^{IPV}_t \approx -0.0085 + 0.00083 t $

which for a projection horizon of 40 years translates to a mean IPV of 2.47% which we then round up to 2.5%. It is plausible of course to consider other percentiles, although, given the size of the sample and the fact that the data is heavily overlapping, the mean is a statistic that is less likely to be misleading; variance measures are likely suppressed. 


It is noted that an AR1 model based on annualised inflation has a different calibration target (its 1 year autoregression, not multi-year variation), so using annual data does not make achieving the IPV target any easier. It is however a second-best model and could be further explored.

The IPV statistic cannot be analytically inferred from the model parameterisation on most occassions; it can be extracted from simulated paths, and as such will always be subject to simulation error, which we can manage by running enough simulations.

As already noted, most time-series models (with an appropriately small and meaningful number of parameters) cannot target a long-horizon IPV;  we can however ajdust our model to target an IPV in several ways.

In this note we have chosen to apply a slow component to the modelled AR1 series. This is the cleanest lever because it boosts annual variance without significantly distorting monthly fit as long as the slow component parameter is close enough to 1. Other possible ways to target IPV that were dropped are as follows: 

| Potential Solution | Reason Dropped |
|--------------------|----------|
| Scale up innovation (residuals) dispersion (multiply NIG scale) | Inflates 1-month variance/tails and overshoots short-horizon behaviour |
| Regime switching (mean and/or variance regimes) with persistent states; also attempted RS on annual NO data | More parameters, harder to identify; does not directly target IPV; annual sample too small and simulated IPV too low |
| Increase persistence in the fast component (raise φ or move from AR(1) to ARMA with near-unit cancellation) | Breaks monthly ACF and forecast fit |



## 5. Final Model specification

After comparing several models the one chosen is an **AR1 seasonally adjusted model with NIG residuals**.
- compared to other monthly models, it offered one of the lowest BIC statistics without introducing too many parameters.
- NIG is appropriate for residuals that are fatter than normal but not as fat as t with small number of dof's.
- models based on annual data either overshoot the IPV statistic (when using NIG residuals), or undershoot it (normal residuals); 
- it is felt that the normal distribution is not appropriate here as its not fat-tailed enough for inflation. 
- IPV was undershoot - however this is not a quantity that is easy to target;

Statistics of the models attempted can be made available on request.

### 5.1 Data
Monthly **log inflation** is defined as

$
r_t = \log(\mathrm{CPI}_t) - \log(\mathrm{CPI}_{t-1}).
$

where $t$ can be any month-end between January 1988 and November 2025. 

### 5.2 Model

The complete model is

$
r_t = \mu + s_{m(t)} + c_t + x_t,
$

where 

- $ \mu$ is the seasonality intercept, 
- $ m(t)\in\{1,\dots,12\}$ denotes the calendar month:
- $ s_2,\dots,s_{12}$ are monthly seasonal offsets, 
- $ x_t = \phi x_{t-1} + \varepsilon_t \,  , \qquad |\phi| < 1$
- $ c_t = \rho c_{t-1} + \eta_t \, , \qquad |\rho| < 1$
- $\varepsilon_t \sim \mathrm{NIG}(a,b,l,s)$, 
- $\eta_t \sim \mathcal{N}(0,\sigma_{\text{slow}}^2), \eta_t$ and $\varepsilon_t$ are independent and i.i.d over time,
- $x_0 = r_T - \mu - s_{11}$ is set as the last filtered value (The last data point is november 2025)
- Initial state for the slow component:  $ c_0 \sim \mathcal{N}\!\left(0,\frac{\sigma_{\text{slow}}^2}{1-\rho^2}\right). $

### 5.3 Estimated parameters:

#### 5.3.1 A-priori parameters
We first allow for seasonality and establish the relevant parameters through OLS:  
$\mu = -0.0044 $

and seasonal dummy coefficients (relative to January (month 1)):

$s_{m(1)} = 0 \\
s_{m(2)}  =     0.009101 \\
s_{m(3)}  =     0.008059 \\
s_{m(4)}  =      0.011631\\
s_{m(5)}  =      0.008047\\
s_{m(6)}  =      0.005766\\
s_{m(7)}  =     0.002492\\
s_{m(8)}  =    0.008392\\
s_{m(9)}  =   0.007668\\
s_{m(10)}  =    0.006467\\
s_{m(11)}  =    0.005513\\
s_{m(12)}  =    0.007455 $

The following parameters are for the model that allows for seasonality and attempts to capture the heavy-tails of the residuals. At this point the model does not target annualised IPV. 

- $\phi \approx 0.25$, the monthly AR1 parameter

The NIG parameters are: 
- $a = 1.2641$
- $b = 0.4514$
- $l = -0.001118$
- $s = 0.002920$

This component captures short-term dynamics and tail behaviour.

#### 5.3.2 Slow component (low-frequency dynamics)
- $\rho = 0.995$, (set arbitrarily to ensure the component has long memory; its half life for monthly sim steps is c11.5 years)
- IPV target: 2.5% $==> \sigma_{\text{slow}} = 0.0002585$ (estimated using trial and error until simulated IPV $\approx$ target IPV)
- $c_0 \sim N\left(0, \frac{\sigma_{\text{slow}}^2}{1 - \rho^2} \right)$  (randomly initialised in each simulation for stationarity)
- Note that all $r_t, x_t, c_t$ are in log-monthly units; IPV is computed on annualised rates.

### 5.3.3 Aggregation and validation
Annual inflation is obtained by aggregation of monthly log inflation:

$
\pi^{(1y)}_t
= \exp\!\left(\sum_{i=1}^{12} r_{t+i}\right) - 1.
$

## 6. Simulated results

Finally we show a couple of charts with key statistics of the simulated series for 100K simulations.  Both charts show key percentiles, mean and median and 5 randomly chosen individual paths (the same ones). The first chart is based on an IPV of 2.076%, which is the average IPV for 37-year long paths. The second is based on an IPV of 2.5%, the extrapolated mean IPV for paths that are 30 to 35 years long. Note that y-axes are on different scales.

<img src="100Ksims_40yrs.png" width="800">
<img src="100Ksims_40yrs2.png" width="800">
<!-- ![Annual inflation fan chart](100Ksims_40yrs.png)
![Annual inflation fan chart](100Ksims_40yrs2.png) -->

It can be seen that in at least 50% of the scenarios, inflation can "escape" the most frequent LPI limits (0% to 5%) for some of the simulation years. The individual paths also illustrate some of the characteristics of past inflation behaviour, such as to spike and return to normal levels in a short period of time. Persistent heteroscedacity is also observed in some scenarios, although it is not noticeable in the historic dataset.

## 7. Interaction with AI

Most of the above models have been built in close conversation with an AI platform. In particular, AI has assisted in: 
- identifying data sources
- cleansing, importing and manipulating the data
- identifying, comparing and contrasting different models for different parts of the process
- providing code for performing the above tasks
- educating the author to the properties and suitability or otherwise of various models

One cannot take AI's responses as correct and critical thinking remains imperative at present (Feb 2026). Below are some thoughts based on the author's journey;
- People will increasinlgy expect you to have "googled" or used AI before you ask them a question. That will make better use of their time and yours.
  - Use AI to 
      - learn programming; AI may get stuck in a loop and become unable to make a very simple code fix that you can do, if you can understand the code. Coding is still a very valuable skill!
      - ask those questions you were afraid to ask, in case others think less of you; but do decide whether this is personal development or good use of company's time.
      - learn something new - you can get significant expertise in an area where you were previously a novice. that will reduce the burden on others who need to train you. 
      - do trivial or repetitive tasks, such as data-scraping
      - proof-read documents and ask for suggestions.

- AI will appear extremely confident on its response and provide a solution to almost any problem. There are very few instances where it will say "I cannot do this". A healthy level of sceptisism can be useful. 
    - In some of the hardest questions, AI can provide a positive answer but will qualify its response. For example, it may say X is possible, but then add caveats. Read the caveats carefully. Consider whether the positive answer provided is meaningful.
    - AI is yet no substitute for reading documents and literature; its summaries are still highly likely to miss crucial points. Nevertheless, AI will point you to many and very useful sources for the kind of work you are involved in.
    - Expect conversations to reach a dead-end on at least a few occasions. Do not let this dishearten you and learn from it.
    - Many times, AI responses are very long and scattergun in nature. AI fails to realise/assess of the significance of each bulletpoint it creates; I had to do that!

- You are driving the work, not AI! Spend time thinking and being imaginative/resourceful, particularly when AI responses drive you to a dead-end. AI will not do that for you. 
- Human subject matter experts are still likely to offer a better response than AI and save you significant time; but not always!
- Do not expect AI to achieve overly complex tasks; aim to provide AI with bitesize instructions; here is an example:
    - Example of bad prompt: _Browse the internet to find me details of inflation swaps and save them in a chart._
    - Example of a better prompt: _I found the website below:  https://pddata.dtcc.com/ppd/cftcdashboard ; in that website there are (cummulative) daily reports of derivative trades; could you write some code to : 
        a/ download all the files in a folder I'll specify; :"SwapData" 
        b/ identify all the GBP inflation swap instruments (the way to do that is find the column labelled UPI FISN (DE in the report I opened) and filter for NA/Swap Infl Idx GBP). 
    - then, identify all the amounts (columns labelled Notional amount-Leg 1 and Notional amount-Leg 2, the inflation rate target (Fixed rate-Leg 1; AU); also keep in all non trivial columns (ones which have more than a single entry or have no blanks)). 
    - then provide me a chart that shows maturity year (Expiration Date, column M) and maturity amount of swaps outstanding. 
    - Note that it is possible that different days may have different number of columns, but if not, please simplify the code._
- Be patient
- Resist the temptation to ask about everything. The 80/20 rule applies here as in any piece of work.

## 8. Summary and conclusions
In this note we constructed an inflation model that can be used for an LPI portfolio to assess capital requirements and assist portfolio optimisation. The model was done in conjunction with AI.

The model is based on historic data, but it is possible to follow a bayesian approach and merge it with current market expectations, as these are illustrated by the derivatives markets.

The key quantity targeted was the Intra-path volatility, the standard deviation of annual inflation rates observed since 1988. This is a key quantity for this kind of modelling, as capital is strongly influenced by the LPI caps and floors of a liability portfolio.

It is imperative for finance practitioners (and in fact for practitioners of any discipline!) to familiarise with and make good use of AI, considering its current capabilities as well as its limitations. AI is a tool that can magnify personal capabilities if used in a reasonable manner. The note concluded with some ideas on best use of AI that reflect the experience of the author.  


## Literature

1. BoE derivation of risk-free nominal and real and break-even inflation curves
https://www.bankofengland.co.uk/-/media/boe/files/working-paper/2001/new-estimates-of-the-uk-real-and-nominal-yield-curves.pdf

2. Mercurio & Morreti paper on SABR vol for inflation. 
https://www.risk.net/sites/default/files/import_unmanaged/risk.net/data/risk/pdf/technical/2009/risk_0609_technical_inflation.pdf

3. Joining the SABR and Libor models together by Mercurio & Morreti
https://www.risk.net/sites/default/files/import_unmanaged/risk.net/data/asiarisk/pdf/2009/asiarisk_may09_cuttingedge.pdf

4. Original SABR paper by Hagan:  MANAGING SMILE RISK - by PATRICK S. HAGAN, DEEP KUMAR, ANDREW S. LESNIEWSKI, AND DIANA E. WOODWARD

5. Belgrade et al paper from 2004 https://shs.hal.science/halshs-03331510/document

6. Open-Gamma papers (Arroub Zine-eddine)
https://quant.opengamma.io/Inflation-Convexity-Adjustment-OpenGamma.pdf
https://quant.opengamma.io/Inflation-Cap-Floor-OpenGamma.pdf

7. Real Rates, Expected inflation and Inflation risk premia, Martin D.D Evans, Journal of Finance, February 1998, pp187-218

8. Inflation-rate Derivatives: From Market Model to Foreign Currency Analogy; Lixin Wu, 2008

9. Paper on quadratic gaussian by Trovato: https://www.citystgeorges.ac.uk/__data/assets/pdf_file/0011/118568/QGYpaper.pdf

10. Paper on volatility surfaces and arbitrage free surfaces: https://www-2.rotman.utoronto.ca/~hull/DownloadablePublications/DHSPaperdraft7.pdf

11. European Actuarial Association paper on inflation risk modelling; general points https://actuary.eu/wp-content/uploads/2023/11/A-Primer-on-Inflation-Risk-Management.pdf?utm_source=chatgpt.com

12. CRO forum - paper on inflation risk management: https://www.thecroforum.org/wp-content/uploads/2024/05/CRO-Forum-Inflation-Risk-paper-1.pdf?utm_source=chatgpt.com

13. Calibration of VAR models with overlapping data; IFoA, Smith, Sharpe, Mehta, etc https://www.cambridge.org/core/services/aop-cambridge-core/content/view/B20D66D81DB918AFD3BBDF9EDAC20863/S1357321719000151a.pdf/calibration_of_var_models_with_overlapping_data.pdf

14. LPI Risk Working party; Alexandra Miles; John Southall (https://www.actuaries.org.uk/system/files/field/document/SessionalApril2019_FINAL.PDF)

15. A regime-switching model on inflation: https://www.soa.org/globalassets/assets/files/research/projects/research-2012-02-effect-deflation-user-guide.pdf

16. Market for inflation instruments: https://www.bankofengland.co.uk/-/media/boe/files/working-paper/2023/the-market-for-inflation-risk.pdf

## Appendix 1 - Seasonality adjustments

Allowing for seasonality is not particularly critical when our objective is annual inflation steps and the question can be stepped aside altogether with annual modelling. In our case we considered two possible models; seasonal dummies (where dummies capture month-specific effects) and seasonal differences (which results by substracting the value 12 months back before fitting a time series).

We preferred the dummies option because it does not introduce moving-average (MA) considerations in the time-series fitting process (also the stochastically-generated paths will display MA traits - we did not find clear evidence from these from the monthly data.)

Below we show three sets of charts; The first one shows the monthly series together with the two options where we have adjusted for seasonal effects; 

<img src="mnthlyCPI_vsSeasRmvd.png" width="800">

We can see that the dummies option deals slightly better with volatility in periods of extreme inflation, by slightly reducing it overall, while the differencing method simply differs it by 12 months. 

Finally we show the ACF and PACF charts before and after applying seasonality adjustments. 

<img src="ACFnPACF_OrigVSDummies.png" width="800">
<img src="ACFnPACF_OrigVSDiffs.png" width="800">

Both charts illustrate significant improvement to effects after 12 months and there is a noticable strongly negative spike at 12 months in the differences option (as expected). While some spikes remain beyond 12 months in the dummies option, it was not deemed worthwhile to increase the complexity of the model to remove them. 

## Appendix 2 - Other potential data sources

Next we have a brief look at the inflation derivatives market, as this gives us insignts on expectations for future inflation. 

The chart below is based on the same data source used by the Bank of England (DTCC Trade repository). We extracted information about transactions over 2025; these are transactions on inflation derivatives, mainly swaps but there are some options as well. 

<img src="swapsbyexpiry2.png" width="800">
<!-- ![Swap contract maturities and volumes](swapsbyexpiry2.png) -->

The chart shows the levels at which inflation is traded at for a range of future years. There are 3 spikes that are clearly visible, in 2022, 2038 and 2045. The one in 2022 is very likely due to a genuine feature - actual inflation has indeed been very high that year, due to Covid and the situation in Ukraine. The spikes in the future, at first look clear outliers. We had a look at the data at 2038 and identified 4 trades where the rate recorded was 3.6925 (when most of the other trades were for a trade between 0.03 and 0.04). So this is quite likely a data error which one should either correct (divide by 100) or exclude from the results. 

Errors aside, one should consider that inflation rates implied by swaps are influenced by supply and demand, as well as risk aversion and many other factors that would not necessarily drive inflation; the volume of the trades (spikes at 2035, 2045 and 2055) indicate attempts to immunise rather than precisely match inflation in a portfolio. While the model we built relies purely on historical data alone, it is not unreasonable to depart from a purely historical model and include aspects of the market views shown in this chart in a capital model (e.g. by using a Bayesian approach).

## Disclaimer 
This document reflects the author’s personal views and has been prepared for discussion and informational purposes only. It does not constitute investment advice, actuarial advice, risk management advice, or a recommendation of any kind.

While care has been taken in the development of the methodology and the analysis presented, no representation or warranty, express or implied, is made as to the accuracy, completeness, or suitability of the information contained herein. Any errors or omissions are the sole responsibility of the author.

No liability is accepted for any loss or damage arising from the use of, or reliance on, this document, including any decision to act or refrain from acting based on its contents.

All data used in this document are sourced from publicly available information. Data sources are acknowledged where relevant in the text and references. Any analysis, interpretation, or conclusions drawn are those of the author.

END